# Part 5 — Feature Engineering
**Appliance Energy Use Forecasting — 7PAM2033**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DolapoMichael/Time-Series-coding-Case-study-and-Report/blob/main/notebooks/05_feature_engineering.ipynb)

Builds the supervised-learning feature table for Part 6's feature-based model: time-of-day/day-of-week encodings, lagged and rolling statistics of the target, plus the sensor/weather columns already in the hourly data.

**On data leakage and "conditional" forecasts:** time-based features are always genuinely known in advance. Lag/rolling features and sensor/weather covariates for the test period, however, are built from real historical data rather than the model's own recursive predictions — so Part 6's model will be a **conditional** forecast (assumes perfect knowledge of future weather and realised past target values), not a blind forecast like SARIMAX in Part 4. That makes the SARIMAX-vs-feature-model comparison in Part 8 not strictly apples-to-apples.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW_CSV_URL = 'https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv'
TARGET = 'Appliances'
LAGS = [1, 2, 3, 6, 12, 24, 48, 168]
ROLLING_WINDOWS = [3, 6, 12, 24, 168]

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
for d in [DATA_DIR, OUTPUT_DIR / 'figures']:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

## Load hourly data (self-contained)

In [ ]:
hourly_path = DATA_DIR / 'energydata_hourly.csv'

if hourly_path.exists():
    hourly = pd.read_csv(hourly_path, index_col=0, parse_dates=True)
else:
    print('No local hourly dataset found — rebuilding from the raw source...')
    raw = pd.read_csv(RAW_CSV_URL)
    raw['date'] = pd.to_datetime(raw['date'])
    raw = raw.set_index('date').sort_index()
    energy_cols = ['Appliances', 'lights']
    sensor_cols = [c for c in raw.columns if c not in energy_cols + ['rv1', 'rv2']]
    hourly = pd.concat([
        raw[energy_cols].resample('h').sum(),
        raw[sensor_cols].resample('h').mean(),
    ], axis=1)
    hourly.to_csv(hourly_path)

print(f'Input hourly shape: {hourly.shape}')

## Feature builders

In [ ]:
def add_time_features(df):
    """Cyclical + categorical timestamp encodings — always known at any
    forecast origin, no matter how far ahead."""
    out = df.copy()
    out['hour'] = out.index.hour
    out['dayofweek'] = out.index.dayofweek
    out['is_weekend'] = (out['dayofweek'] >= 5).astype(int)
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
    out['dow_sin'] = np.sin(2 * np.pi * out['dayofweek'] / 7)
    out['dow_cos'] = np.cos(2 * np.pi * out['dayofweek'] / 7)
    return out

def add_lag_features(df, target, lags=LAGS):
    """Lagged target values — lag_N at row t is the value N hours before t."""
    out = df.copy()
    for lag in lags:
        out[f'lag_{lag}'] = out[target].shift(lag)
    return out

def add_rolling_features(df, target, windows=ROLLING_WINDOWS):
    """Rolling mean/std of the target. shift(1) before rolling() is essential
    — without it the window for row t would include t's own target value,
    the single most common leakage bug in time-series feature engineering."""
    out = df.copy()
    for window in windows:
        shifted = out[target].shift(1)
        out[f'roll_mean_{window}'] = shifted.rolling(window).mean()
        out[f'roll_std_{window}'] = shifted.rolling(window).std()
    return out

def build_feature_table(hourly, target=TARGET):
    out = add_time_features(hourly)
    out = add_lag_features(out, target)
    out = add_rolling_features(out, target)
    return out.dropna()

## Build the feature table

In [ ]:
features = build_feature_table(hourly, TARGET)
print(f'Feature table shape after dropna: {features.shape}  '
      f'({hourly.shape[0] - features.shape[0]} rows dropped for lag/rolling warm-up)')

time_cols = ['hour', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
lag_cols = [f'lag_{l}' for l in LAGS]
rolling_cols = [c for c in features.columns if c.startswith('roll_')]
sensor_weather_cols = [c for c in hourly.columns if c != TARGET and c != 'lights']

print(f'\nTime-based     ({len(time_cols)}): {time_cols}')
print(f'Lag            ({len(lag_cols)}): {lag_cols}')
print(f'Rolling        ({len(rolling_cols)}): {rolling_cols}')
print(f'Sensor/weather ({len(sensor_weather_cols)}): {sensor_weather_cols}')

features.to_csv(DATA_DIR / 'energydata_features.csv')
print(f"\nSaved to {DATA_DIR / 'energydata_features.csv'}")

## Correlation with the target

In [ ]:
corr = features.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False)
top = corr.head(20)

fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#c0392b' if v < 0 else '#1f5b8a' for v in top.values]
ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1])
ax.set_xlabel(f'Correlation with {TARGET}')
ax.set_title('Top 20 engineered features by |correlation| with Appliances')
ax.axvline(0, color='black', linewidth=0.8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '05_feature_correlations.png')
plt.show()

print('Top 10:')
corr.head(10).round(3)